## Data Preparation and Priority Mapping
First, we address the $q_j$ explosion. scheduling_class ranges from 0-3, which is dangerous because a priority of $0$ mathematically nullifies the customer's satisfaction ($0 \cdot v = 0$). We will map both options into strict, non-zero ordinal tiers $\{1, 2, 3, 4, 5\}$.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import linprog

def apply_priority_mapping(df: pd.DataFrame, mode: str = 'tiered') -> pd.DataFrame:
    """
    Maps the raw priority to prevent the satisfaction objective from blowing up.
    """
    df_mapped = df.copy()
    
    if mode == 'raw':
        # Baseline: No mapping, raw values (0, 118, 103, 116, 105, 107, 200, 119, 115, 25, 360) 
        # We add +1 so priority 0 doesn't zero out satisfaction entirely
        df_mapped['q_j'] = df_mapped['q_j'] + 1 
        
    elif mode == 'scheduling_class':
        # Scheduling class is typically 0, 1, 2, 3 in Google traces. 
        # Shift to 1, 2, 3, 4 to prevent zero-multiplication.
        df_mapped['q_j'] = df_mapped['scheduling_class'] + 1
        
    elif mode == 'tiered':
        # Map raw priorities to 5 standard tiers based on Google Trace documentation
        bins = [-1, 99, 115, 119, 350, float('inf')]
        labels = [1, 2, 3, 4, 5] 
        # 1: Best Effort, 2: Batch, 3: Mid-tier, 4: Production, 5: Latency-Critical
        df_mapped['q_j'] = pd.cut(df_mapped['q_j'], bins=bins, labels=labels).astype(float)
        
    else:
        raise ValueError("Unknown mapping mode.")
        
    return df_mapped

# Example usage (assuming batch_df is already loaded in your notebook)
# batch_tiered = apply_priority_mapping(batch_df, mode='tiered')

## Normalized optimizer class
During initialization, it automatically computes the ideal single-objective maximums ($z^*$) and the worst-case Carbon Cost. It then uses these values as denominators to scale everything into a strict $[0, 1]$ range.

In [2]:
def build_fluid_volume_constraints(jobs_df, capacities, horizon):
    A_cpu = jobs_df['A_cpu'].values.astype(float) * jobs_df['D (hours)'].values.astype(float)
    A_ram = jobs_df['A_ram'].values.astype(float) * jobs_df['D (hours)'].values.astype(float)
    A_ub = np.vstack([A_cpu, A_ram])
    b_ub = np.array([capacities['cpu'] * horizon, capacities['ram'] * horizon])
    return A_ub, b_ub

class NormalizedStaticOptimizer:
    def __init__(self, jobs_df: pd.DataFrame, capacities: dict, horizon: float = 744.0, normalize: bool = True):
        self.jobs_df = jobs_df.copy()
        self.n = len(self.jobs_df)
        self.normalize = normalize
        
        self.A_ub, self.b_ub = build_fluid_volume_constraints(self.jobs_df, capacities, horizon)
        
        # Raw coefficients
        self.q_j = self.jobs_df['q_j'].values
        self.v_total = self.jobs_df['v_total'].values
        self.phi_total = self.jobs_df['phi_total'].values
        self.C_elec = self.jobs_df['C_elec'].values
        self.C_carbon = self.jobs_df['C_carbon'].values
        
        self.c_sat = self.q_j * self.v_total
        self.c_prof = self.phi_total - self.C_elec
        self.c_carb = self.C_carbon  # Carbon Cost (we want to minimize this)

        self._compute_ideal_and_nadir()

    def _compute_ideal_and_nadir(self):
        """Calculates the absolute maximums for normalization denominators."""
        bounds = [(0, 1)] * self.n
        
        # 1. Max Satisfaction
        res_sat = linprog(-self.c_sat, A_ub=self.A_ub, b_ub=self.b_ub, bounds=bounds, method='highs')
        self.z_sat_max = -res_sat.fun if res_sat.status == 0 else 1.0
        
        # 2. Max Profit
        res_prof = linprog(-self.c_prof, A_ub=self.A_ub, b_ub=self.b_ub, bounds=bounds, method='highs')
        self.z_prof_max = -res_prof.fun if res_prof.status == 0 else 1.0
        
        # 3. Max Carbon Cost (Nadir for Sustainability)
        # To find the absolute worst carbon expenditure, we maximize carbon cost
        res_carb = linprog(-self.c_carb, A_ub=self.A_ub, b_ub=self.b_ub, bounds=bounds, method='highs')
        self.z_carb_max = -res_carb.fun if res_carb.status == 0 else 1.0
        
        print(f"Computed Limits -> Sat_Max: ${self.z_sat_max:,.0f} | Prof_Max: ${self.z_prof_max:,.0f} | Carb_Max: ${self.z_carb_max:,.0f}")

    def linear_scalarization(self, l1, l2, l3):
        """Standard LP with optional Min-Max Normalization."""
        if self.normalize:
            # Divide each coefficient by its maximum possible outcome
            r_j = (l1 * (self.c_sat / self.z_sat_max) + 
                   l2 * (self.c_prof / self.z_prof_max) - 
                   l3 * (self.c_carb / self.z_carb_max))
        else:
            r_j = l1 * self.c_sat + l2 * self.c_prof - l3 * self.c_carb
            
        res = linprog(-r_j, A_ub=self.A_ub, b_ub=self.b_ub, bounds=[(0,1)]*self.n, method='highs')
        x = np.round(res.x)
        return self._evaluate_solution(x)

    def chebyshev_scalarization(self, l1, l2, l3):
        """
        Normalized Chebyshev: Minimizes relative distance to Utopian point.
        Distance constraints: lambda * (z_ideal - f(x)) / (z_ideal - z_nadir) <= t
        """
        # Objective: minimize t
        c = np.append(np.zeros(self.n), 1.0) 
        
        rows, rhs = [], []
        
        if self.normalize:
            # Sat: lambda * (1 - f_sat/z_sat_max) <= t  ==>  -lambda * (c_sat/z_sat_max)*x - t <= -lambda
            rows.append(np.append(-l1 * (self.c_sat / self.z_sat_max), -1.0))
            rhs.append(-l1)
            
            # Prof: lambda * (1 - f_prof/z_prof_max) <= t ==> -lambda * (c_prof/z_prof_max)*x - t <= -lambda
            rows.append(np.append(-l2 * (self.c_prof / self.z_prof_max), -1.0))
            rhs.append(-l2)
            
            # Carb: Ideal is 0. Nadir is z_carb_max. Distance = (C_carb - 0) / (z_carb_max - 0)
            # lambda * (c_carb/z_carb_max)*x - t <= 0
            rows.append(np.append(l3 * (self.c_carb / self.z_carb_max), -1.0))
            rhs.append(0.0)
        else:
            rows.append(np.append(-l1 * self.c_sat, -1.0)); rhs.append(-l1 * self.z_sat_max)
            rows.append(np.append(-l2 * self.c_prof, -1.0)); rhs.append(-l2 * self.z_prof_max)
            rows.append(np.append(l3 * self.c_carb, -1.0)); rhs.append(0.0)
            
        # Add capacity constraints
        for i in range(self.A_ub.shape[0]):
            rows.append(np.append(self.A_ub[i], 0.0))
            rhs.append(self.b_ub[i])
            
        res = linprog(c, A_ub=np.vstack(rows), b_ub=np.array(rhs), bounds=[(0,1)]*self.n + [(0, None)], method='highs')
        x = np.round(res.x[:self.n])
        return self._evaluate_solution(x)

    def _evaluate_solution(self, x):
        return {
            'V_sat': float(np.sum(x * self.c_sat)),
            'V_prof': float(np.sum(x * self.c_prof)),
            'C_carbon': float(np.sum(x * self.c_carb))
        }

## Running the Matrix Experiment
This cell sweeps through the different priority modes and normalizations.

In [8]:
# Assuming your capacities and horizon are defined

DATA_PATH = "../data/processed/batch_may2019_2k.csv" 

batch_df = pd.read_csv(DATA_PATH)


CLUSTER_CAPACITY = {'cpu': 285.0, 'ram': 162.0}
HORIZON_HOURS = 744.0
EQUAL_WEIGHTS = (0.333, 0.333, 0.333)

experiments = []

for p_mode in ['raw', 'scheduling_class', 'tiered']:
    print(f"\n--- Testing Priority Mode: {p_mode.upper()} ---")
    df_mapped = apply_priority_mapping(batch_df, mode=p_mode)
    
    for norm_flag in [False, True]:
        opt = NormalizedStaticOptimizer(df_mapped, CLUSTER_CAPACITY, HORIZON_HOURS, normalize=norm_flag)
        norm_str = "Normalized" if norm_flag else "Unnormalized"
        
        # Test Linear Scalarization
        lin_res = opt.linear_scalarization(*EQUAL_WEIGHTS)
        
        # Test Chebyshev Scalarization
        cheb_res = opt.chebyshev_scalarization(*EQUAL_WEIGHTS)
        
        experiments.append({
            'Priority_Mode': p_mode,
            'Normalization': norm_str,
            'Sat_Max_Potential': opt.z_sat_max,
            'Linear_V_sat': lin_res['V_sat'],
            'Linear_V_prof': lin_res['V_prof'],
            'Cheb_V_sat': cheb_res['V_sat'],
            'Cheb_V_prof': cheb_res['V_prof'],
        })

# Display Results
results_df = pd.DataFrame(experiments)

# Create a dictionary specifying how each column should be formatted
format_dict = {
    'Sat_Max_Potential': '${:,.0f}',   # No decimals for the Utopian limit
    'Linear_V_sat':      '${:,.2f}',   # 2 decimals for the actual results
    'Linear_V_prof':     '${:,.2f}',
    'Cheb_V_sat':        '${:,.2f}',
    'Cheb_V_prof':       '${:,.2f}'
}

# Apply the style and display
display(results_df.style.format(format_dict))


--- Testing Priority Mode: RAW ---
Computed Limits -> Sat_Max: $7,583,071 | Prof_Max: $277,487 | Carb_Max: $21,034
Computed Limits -> Sat_Max: $7,583,071 | Prof_Max: $277,487 | Carb_Max: $21,034

--- Testing Priority Mode: SCHEDULING_CLASS ---
Computed Limits -> Sat_Max: $603,656 | Prof_Max: $277,487 | Carb_Max: $21,034
Computed Limits -> Sat_Max: $603,656 | Prof_Max: $277,487 | Carb_Max: $21,034

--- Testing Priority Mode: TIERED ---
Computed Limits -> Sat_Max: $519,364 | Prof_Max: $277,487 | Carb_Max: $21,034
Computed Limits -> Sat_Max: $519,364 | Prof_Max: $277,487 | Carb_Max: $21,034


,Priority_Mode,Normalization,Sat_Max_Potential,Linear_V_sat,Linear_V_prof,Cheb_V_sat,Cheb_V_prof
0,raw,Unnormalized,"$7,583,071","$7,582,300.03","$260,694.15","$7,553,449.36","$258,703.99"
1,raw,Normalized,"$7,583,071","$7,422,056.01","$185,549.75","$4,603,861.28","$169,242.88"
2,scheduling_class,Unnormalized,"$603,656","$595,269.69","$274,006.20","$586,513.78","$270,659.55"
3,scheduling_class,Normalized,"$603,656","$570,471.81","$276,571.49","$368,292.76","$166,666.27"
4,tiered,Unnormalized,"$519,364","$514,252.25","$274,364.73","$510,775.25","$262,700.34"
5,tiered,Normalized,"$519,364","$503,977.55","$275,470.76","$294,609.91","$155,844.96"


`Sat_Max` is the Ideal Point ($z^*_{sat}$) for Customer Satisfaction. It represents the absolute maximum amount of satisfaction your cloud cluster could theoretically generate if the provider cared 100% about customer SLAs and 0% about profit or the environment.

Before running any multi-objective scalarizations, the N`ormalizedStaticOptimizer` runs a Single-Objective Linear Program (LP).Mathematically, it solves:
$$\text{Sat\_Max} = \max_{\mathbf{x} \in \mathcal{X}} \sum_{j=1}^N x_j (q_j \cdot v_j)$$

- `Sat_Max` drops drastically from $7.58M (Raw) to $519k (Tiered). This proves that the raw Google priorities ($q_j \in [0, 360]$) were artificially inflating the maximum satisfaction. By putting them into Tiers ($1 \to 5$), we successfully grounded the metric.

- Normalized Linear: $\max \left( \lambda_1 \frac{f_{sat}}{\text{Sat\_Max}} + \lambda_2 \frac{f_{prof}}{\text{Prof\_Max}} - \lambda_3 \frac{f_{carb}}{\text{Carb\_Max}} \right)$ 
- Normalized Chebyshev: $\min \max \left\{ \lambda_1 \left(1 - \frac{f_{sat}}{\text{Sat\_Max}}\right), \lambda_2 \left(1 - \frac{f_{prof}}{\text{Prof\_Max}}\right), \dots \right\}$

**Row 5 (Normalized Chebyshev)**: Look at Cheb_V_sat ($294,609) and Cheb_V_prof ($155,844). This is the most mathematically honest row in the entire table. Let's calculate the percentages:
- Satisfaction achieved: $294,609 / 519,364 = \mathbf{56.7\%}$ of ideal.
- Profit achieved: $155,844 / 277,487 = \mathbf{56.1\%}$ of ideal.